In [62]:
import pandas as pd
import matplotlib.pyplot as plt
import requests
from datetime import datetime
from urllib.parse import parse_qs
import sqlalchemy
import json
import numpy as np

print('OK')

OK


In [53]:
#url = 'https://okx.com/api/v5/market/history-candles?instId=BTC-USDT&bar=15m&limit=5'
#r = requests.get(url)
#jdata = r.json()['data'][0]

def convert_request_data_to_dict(data, url):
    parsed = requests.utils.urlparse(url)
    query_params = parse_qs(parsed.query)
    asset, interval = query_params.get('instId', [None])[0], query_params.get('bar', [None])[0]
    res_dict = {'asset': asset,
                'interval': interval,
                'timestamp': pd.to_datetime(int(data[0]), utc=False, unit='ms'),
                'open': data[1],
                'high': data[2],
                'low': data[3],
                'close': data[4],
                'volume': data[5]}
    return res_dict


In [ ]:
def get_candles(asset, bar, limit):
    url = f'https://okx.com/api/v5/market/history-candles?instId={asset}&after=1704067200000&bar={bar}&limit={limit}'
    r = requests.get(url)
    candles = []
    for i in range(int(limit)):
        jdata = r.json()['data'][i]
        candle = convert_request_data_to_dict(jdata, url)
        candles.append(candle)
    return candles


get_candles('BTC-USDT', '4H', '300')

# this code only gets first 300 candles per request, but i can get all candles, how?

[{'asset': 'BTC-USDT',
  'interval': '4H',
  'timestamp': Timestamp('2023-12-31 20:00:00'),
  'open': '42615.8',
  'high': '42680.8',
  'low': '42066',
  'close': '42283',
  'volume': '1232.51791673'},
 {'asset': 'BTC-USDT',
  'interval': '4H',
  'timestamp': Timestamp('2023-12-31 16:00:00'),
  'open': '42457.6',
  'high': '42720',
  'low': '42430.1',
  'close': '42612.1',
  'volume': '726.90376847'},
 {'asset': 'BTC-USDT',
  'interval': '4H',
  'timestamp': Timestamp('2023-12-31 12:00:00'),
  'open': '42522',
  'high': '42658.5',
  'low': '42346',
  'close': '42457.5',
  'volume': '1337.70932786'},
 {'asset': 'BTC-USDT',
  'interval': '4H',
  'timestamp': Timestamp('2023-12-31 08:00:00'),
  'open': '42539.7',
  'high': '42888',
  'low': '42369.5',
  'close': '42522',
  'volume': '2341.12243703'},
 {'asset': 'BTC-USDT',
  'interval': '4H',
  'timestamp': Timestamp('2023-12-31 04:00:00'),
  'open': '42174.1',
  'high': '42678.9',
  'low': '42153',
  'close': '42539.5',
  'volume': '1302

In [ ]:
def convert_to_timestamp(datetime_str):
    res = datetime.strptime(datetime_str, "%Y-%m-%d %H:%M:%S").timestamp()
    return res

def convert_dt_for_interval(dt, interval): # currently up to 4H, >= 1d doesnt work
    # '2020-01-12 17:48:39', '15m'
    # returns -> '2020-01-12 17:45:00'
    # how?
    # mb like this?
    # u can use s, m, H
    d, t = dt[:10], dt[11:]
    trim_value, trim_dim = int(interval[:-1]), interval[-1]
    match trim_dim:
        case 's':
           new_s = str((int(t[-2:]) // trim_value) * trim_value)
           t = t[:-2] + new_s
        case 'm':
            new_m = str((int(t[-5:-3]) // trim_value) * trim_value)
            t = t[:-5] + new_m + ':00'
        case 'H':
            new_H = str((int(t[-8:-6]) // trim_value) * trim_value)
            t = t[:-8] + new_H + ':00:00'
    new_dt = d + ' ' + t
    return new_dt


def fetch_all_candles(asset, bar, limit, start_time=None, end_time=None):
    end_time = convert_to_timestamp('2018-01-12 00:00:00')
    current_datetime = str(datetime.now())[:-7]
    start_time = convert_to_timestamp(current_datetime)
    print(start_time)
    url = f'https://okx.com/api/v5/market/history-candles?instId={asset}&bar={bar}&limit={limit}'

fetch_all_candles(1,1,1)
convert_dt_for_interval('2020-01-12 17:48:39', '4H')

1771775867.0


'2020-01-12 16:00:00'